In [11]:
import torch
from typing import Any

class MyModelLN(torch.nn.Module):
    class Block(torch.nn.Module):
        def __init__(self, in_channels, out_channels) -> None:  # sửa thụt lề vào trong Block
            super().__init__()
            self.linear = torch.nn.Linear(in_channels, out_channels)
            self.norm = torch.nn.LayerNorm(out_channels)
            self.relu = torch.nn.ReLU()
            if in_channels != out_channels:
                self.skip = torch.nn.Linear(in_channels, out_channels)
            else:
                self.skip = torch.nn.Identity() 

        def forward(self, x) -> Any:  # sửa thụt lề vào trong Block
            return self.relu(self.norm(self.linear(x))) + self.skip(x)  # sửa + x -> + self.skip(x)

    def __init__(self, layer_size=[512, 512, 512]) -> None:
        super().__init__()
        layers = []
        layers.append(torch.nn.Flatten())
        c = 128 * 128 * 3
        layers.append(torch.nn.Linear(c, 512, bias=False))
        c = 512
        for s in layer_size:
            layers.append(self.Block(c, s))  # dùng Block có residual connection
            c = s
        layers.append(torch.nn.Linear(c, 102, bias=False))  # sửa dấu ngoặc: đưa bias=False vào trong Linear(...)
        self.model = torch.nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> Any:
        return self.model(x)

x = torch.randn(10,3,128,128)
netn = MyModelLN([512]*4)
print(netn(x))

tensor([[-0.7182, -0.0753,  1.0112,  ..., -1.0055,  0.4303,  1.6440],
        [ 0.5028, -0.4267,  2.1536,  ...,  2.1588, -0.4524,  0.4576],
        [-1.1115,  0.5090,  1.8993,  ..., -0.1330, -0.5587,  0.4271],
        ...,
        [-0.6769, -0.7105,  1.1973,  ...,  0.0888, -0.6035,  0.0069],
        [-0.3224, -1.4219,  1.7460,  ...,  0.1053, -0.7499, -0.8676],
        [-1.1645, -1.5707,  2.0685,  ...,  0.2284,  0.1502, -0.2185]],
       grad_fn=<MmBackward0>)


In [10]:
import torch
from typing import Any

class MyModelLN(torch.nn.Module):
    class Block(torch.nn.Module):
        def __init__(self, in_channels, out_channels) -> None:  # sửa thụt lề vào trong Block
            super().__init__()
            self.model = torch.nn.Sequential(
                torch.nn.Linear(in_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
                torch.nn.Linear(in_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU()
            )
            if in_channels != out_channels:
                self.skip = torch.nn.Linear(in_channels, out_channels)
            else:
                self.skip = torch.nn.Identity() 

        def forward(self, x) -> Any:  # sửa thụt lề vào trong Block
            return self.model(x) + self.skip(x)  # sửa + x -> + self.skip(x)

    def __init__(self, layer_size=[512, 512, 512]) -> None:
        super().__init__()
        layers = []
        layers.append(torch.nn.Flatten())
        c = 128 * 128 * 3
        layers.append(torch.nn.Linear(c, 512, bias=False))
        c = 512
        for s in layer_size:
            layers.append(self.Block(c, s))  # dùng Block có residual connection
            c = s
        layers.append(torch.nn.Linear(c, 102, bias=False))  # sửa dấu ngoặc: đưa bias=False vào trong Linear(...)
        self.model = torch.nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> Any:
        return self.model(x)

x = torch.randn(10,3,128,128)
netn = MyModelLN([512]*4)
print(netn(x))

tensor([[-3.3922e+00,  6.2093e-01,  1.5418e+00,  ...,  1.9902e+00,
          1.2433e-01, -6.4079e-01],
        [-2.3171e+00,  1.3563e+00,  6.6548e-01,  ...,  2.2301e+00,
         -6.1328e-01, -1.3628e+00],
        [-2.3391e+00,  1.2450e+00,  2.1150e+00,  ...,  9.0370e-01,
         -1.5654e-01, -8.2268e-02],
        ...,
        [-2.3941e+00,  1.3926e+00,  1.2388e+00,  ...,  1.6542e+00,
         -2.9979e-01, -3.3119e-01],
        [-3.1816e+00,  3.4256e-01, -1.8420e-02,  ...,  1.2621e+00,
          3.3013e-03, -4.7900e-01],
        [-3.1434e+00,  1.4410e+00,  1.8260e+00,  ...,  9.2311e-01,
          4.6197e-01, -5.6435e-01]], grad_fn=<MmBackward0>)
